In [ ]:
# 安裝套件
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
# 從 Colab 安全儲存區讀取 API / Token
from google.colab import userdata

ngrok_authtoken = "3ClDmzH22sqWEbRSdze4yo4Zgpf_7ycL9z7EcZyfCYuL92PAR"
line_channel_access_token = "1OQmRLQRLLrqhossKfNHRNPO7oKJBICehSGz0ZmTf/I3JuDjHP2cYZX9L8u/MRrHuO6MmNe+szkjDQSHLoKreAvJxy/k7n0QhuWs/M31wA6meIcAA7nMmcPUsi9VR2SWtKdL3ny/TSnTrMKfBml3CwdB04t89/1O/w1cDnyilFU="
line_channel_secret = "79de232aeab6cf1e0df8c7723791c4bc"
gemini_api_key = "xxxxx"

port = 5051  # Flask 埠號

In [ ]:
# ngrok
from pyngrok import ngrok
import requests

ngrok.kill()
ngrok.set_auth_token(ngrok_authtoken)

tunnel = ngrok.connect(port)
webhook_url = tunnel.public_url

print("Webhook URL:", webhook_url)

In [ ]:
# 更新 LINE webhook
def update_line_webhook(webhook_url):

    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"

    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }

    data = {"endpoint": webhook_url}

    response = requests.put(url, headers=headers, json=data)

    print("status:", response.status_code)
    print("response:", response.text)

update_line_webhook(webhook_url)

In [ ]:
# Gemini 初始化（stateful）
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
    google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
# stateful（有記憶）查詢函式
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
# 測試（0704重點）
result = stateful_query("簡介明新科技大學")
print(result)

result2 = stateful_query("校長是誰？")
print(result2)

# 為什麼「校長是最新的」：
# 因為 stateful chat 會記住前面的「明新科技大學」
# 所以後面問校長時會自動延續上下文
# 判斷為「該校最新校長資訊」

In [8]:
# Flask + LINE Bot
from flask import Flask, request, abort
from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():

    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)

    print("BODY:", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]

            reply_text = stateful_query(prompt)
            if not reply_text:
                reply_text = "無法取得回覆，請稍後再試"

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


if __name__ == "__main__":
    app.run(port=port)# Flask + LINE Bot
from flask import Flask, request, abort
from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():

    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)

    print("BODY:", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):

    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


if __name__ == "__main__":
    app.run(port=port)